In [349]:
include("../src/EBC.jl")
include("../../BeamToyModel/src/BeamToyModel.jl")
using Base.Threads
using Statistics
using NPZ

In [350]:
nside_in = 32
res = Resolution(nside_in)
lmax_in = 3nside_in-1
fwhm = deg2rad(1.0)
blm_harmonic = BeamToyModel.gaussian_harmonic(lmax_in, fwhm, mmax = lmax_in)
;

In [351]:
cs = ConvolutionSky()
fieldnames(typeof(cs))
cb = ConvolutionBeam()
fieldnames(typeof(cb))
cc = ConvolutionCalculate(nside_output = nside_in, lstart = 0, mmax_calculate = 2)
fieldnames(typeof(cc))

(:nside_output, :lstart, :lstop, :mmax_calculate, :HWP)

In [352]:
nalm = Healpix.numberOfAlms(lmax_in, lmax_in)
almT = rand(ComplexF64, nalm)
almE = rand(ComplexF64, nalm)
almB = rand(ComplexF64, nalm)
cs.lmax = lmax_in
cs.alm = hcat(almT, almE, almB)
alm = npzread("alm=CMB=r0=nside32=seed1.npy")
alm_smooth = npzread("alm_smooth=CMB=r0=nside32=seed1.npy")
cs.alm = hcat(alm[1,:], alm[2,:], alm[3,:])
;

In [353]:
cb.lmax = lmax_in
cb.mmax = 2
cb.blm = hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm)
;

In [354]:
pix_idx = 6
θ_pix,φ_pix = Healpix.pix2angRing(res, pix_idx)

(0.05103657515266638, 1.1780972450961724)

In [355]:
nptg = 1
θ = θ_pix
φ = φ_pix
dθ = rand(nptg)*1e-2*0
dφ = rand(nptg)*1e-2*0
ψ = rand(nptg)*2pi*0

1-element Vector{Float64}:
 0.0

In [356]:
alphas = zeros(size(dθ))
betas = zeros(size(dθ))
gammas = zeros(size(dθ))

for i in eachindex(dθ)
    err, (alphas[i], betas[i], gammas[i]) = check_split(φ, θ, dφ[i], dθ[i], ψ[i])
    if err >= 1e0
        @show err
    end
end

err = 1.8477590650225735


In [357]:
@time A = local_effective_wignerD(cb, cc, alphas, betas, gammas)


  0.289353 seconds (112.04 k allocations: 11.112 MiB, 64.61% compilation time)


9216×474 SparseMatrixCSC{ComplexF64, Int64} with 46053 stored entries:
⎡⢧⠀⠀⎤
⎢⢸⠀⠀⎥
⎢⢸⠀⠀⎥
⎢⢸⡄⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⢿⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⡀⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎣⠀⠀⡇⎦

In [358]:
@inline pm(m::Int, n::Int) = isodd(m - n) ? -1.0 : 1.0

function local_effective_wignerD_conj(cb, cc, α, β, γ)
    n_beam = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)
    n_sky = sum(2*l + 1 for l in cc.lstart:cc.lstop)
    D_beam = spzeros(ComplexF64, n_sky, n_beam)
    @inbounds for l in cc.lstart:cc.lstop
        n_ = min(l, cb.mmax)
        @inbounds for i in eachindex(α)
            @inbounds for m in -l:l
                m_idx = lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cc.lstop)
                @inbounds for n in -n_:n_
                    sgn = pm(m, n)
                    n_idx = lmr_idx(l=l, m=n, lstart=cc.lstart, mmax=cb.mmax)
                    D_beam[m_idx, n_idx] += sgn * WignerD.wignerDjmn(l, -m, -n, α[i], β[i], γ[i])
                end
            end
        end
    end
    return D_beam./length(α)
end

function local_effective_wignerD_conj_reduced(cb, cc, α, β, γ; τ::Int=5)
    n_beam = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)
    n_sky = sum(2*l + 1 for l in cc.lstart:cc.lstop)
    D_beam = spzeros(ComplexF64, n_sky, n_beam)
    @inbounds for l in cc.lstart:cc.lstop
        n_ = min(l, cb.mmax)
        @inbounds for i in eachindex(α)
            @inbounds for m in -l:l
                m_idx = lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cc.lstop)
                # ★帯制限：|m-n| <= τ だけ回す
                n_lo = max(-n_, m - τ)
                n_hi = min( n_, m + τ)
                #@show n_lo, n_hi, n_
                @inbounds for n in n_lo:n_hi
                    sgn = pm(m, n)
                    n_idx = lmr_idx(l=l, m=n, lstart=cc.lstart, mmax=cb.mmax)
                    D_beam[m_idx, n_idx] += sgn * WignerD.wignerDjmn(l, -m, -n, α[i], β[i], γ[i])
                end
            end
        end
    end
    return D_beam./length(α)
end

function global_wignerD_conj(cc, φ, θ, ψ)
    n_sky = sum(2*l + 1 for l in cc.lstart:cc.lstop)
    D_beam = spzeros(ComplexF64, n_sky, n_sky)
    for l in cc.lstart:cc.lstop
        m_ = l
        @inbounds for m in -m_:m_
            m_idx = lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cc.lstop)
            @inbounds for n in -m_:m_
                sgn = pm(m, n)
                n_idx = lmr_idx(l=l, m=n, lstart=cc.lstart, mmax=cc.lstop)
                D_beam[m_idx, n_idx] = sgn * WignerD.wignerDjmn(l, -m, -n, φ, θ, ψ)
            end
        end
    end
    return D_beam
end

global_wignerD_conj (generic function with 1 method)

In [359]:
@time A_conj = local_effective_wignerD_conj(cb, cc, alphas, betas, gammas)


  0.340000 seconds (106.90 k allocations: 10.725 MiB, 50.92% compilation time)


9216×474 SparseMatrixCSC{ComplexF64, Int64} with 46053 stored entries:
⎡⢧⠀⠀⎤
⎢⢸⠀⠀⎥
⎢⢸⠀⠀⎥
⎢⢸⡄⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⢿⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⡀⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎣⠀⠀⡇⎦

In [360]:
@time A2_conj = local_effective_wignerD_conj_reduced(cb, cc, alphas, betas, gammas; τ=20)


  0.204980 seconds (110.14 k allocations: 8.811 MiB, 84.44% compilation time)


9216×474 SparseMatrixCSC{ComplexF64, Int64} with 17544 stored entries:
⎡⢧⠀⠀⎤
⎢⢸⠀⠀⎥
⎢⢸⠀⠀⎥
⎢⢸⡀⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⢳⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢘⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢰⠀⎥
⎢⠀⠸⠀⎥
⎢⠀⠨⠀⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⠇⎥
⎣⠀⠀⠅⎦

In [361]:
@show maximum(abs.(A_conj .- conj.(A)))
@show maximum(abs.(A2_conj .- conj.(A)))
@show maximum(abs.(A_conj .- (A2_conj) ))

maximum(abs.(A_conj .- conj.(A))) = 2.376180849306131e-14
maximum(abs.(A2_conj .- conj.(A))) = 2.376180849306131e-14
maximum(abs.(A_conj .- A2_conj)) = 1.239768921204587e-14


1.239768921204587e-14

In [362]:
B = global_wignerD_conj(cc, φ, θ, 0)

9216×9216 SparseMatrixCSC{ComplexF64, Int64} with 1179615 stored entries:
⎡⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠈⠻⢆⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠿⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠿⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠿⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠿⣧⣀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⠻⣦⡄⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢻⣶⣀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⠻⣦⡄⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢻⣶⎦

In [363]:
blm_ = blm_lrange(cb,cc)
alm_ = alm_lrange(cs,cc)

3×9216 Matrix{ComplexF64}:
  0.0+0.0im  -0.0+0.0im   0.0+0.0im   0.0+0.0im  …      0.24645+1.48487im
 -0.0-0.0im   0.0-0.0im  -0.0-0.0im  -0.0-0.0im     -0.00706098+0.00788036im
 -0.0-0.0im   0.0-0.0im  -0.0-0.0im  -0.0-0.0im     -0.00473882+0.00797887im

In [364]:
ψ

1-element Vector{Float64}:
 0.0

In [365]:
V = B*A2_conj * blm_
@time res_ = [sum(alm_[k, :] .*  V[:, k]) for k in 1:3]

  0.083994 seconds (21.50 k allocations: 2.708 MiB, 87.18% compilation time)


3-element Vector{ComplexF64}:
   -85.66623121030932 + 5.129430307096474e-13im
 0.053647104400149054 + 0.36441048229566186im
  0.05364710440013382 - 0.3644104822956679im

In [366]:
alm_[2, :]

9216-element Vector{ComplexF64}:
                  -0.0 - 0.0im
                   0.0 - 0.0im
                  -0.0 - 0.0im
                  -0.0 - 0.0im
  -0.11194407259046876 + 0.06209311450486262im
  -0.08818777083472878 + 0.186775321019867im
    0.2161981237558695 + 0.0018863793562230863im
   0.08699683911218295 + 0.18543610655059595im
  -0.11432898454053267 - 0.06414051364986957im
   0.04911347828791041 - 0.015406895261353342im
  0.008412994635545809 + 0.025025564671368645im
  -0.10326162485006961 - 0.031900144801652396im
   -0.1368615430095226 + 0.0010125543843854975im
                       ⋮
 -0.002016181700562887 + 0.016500247591356324im
   0.01502746165699564 + 0.0028599079248389355im
  0.007246673515052047 - 0.022802784241375818im
  -0.03905859192717052 - 0.010745211411481359im
 0.0037159021618096417 + 0.003375599681657909im
 -0.009598740731295564 + 0.019735774811128887im
 -0.021806191927135062 - 0.009843824750003782im
  0.015029991910816899 + 0.027302556306743355im
 -0.0

In [367]:
alm_[3, :]

9216-element Vector{ComplexF64}:
                  -0.0 - 0.0im
                   0.0 - 0.0im
                  -0.0 - 0.0im
                  -0.0 - 0.0im
  -0.11432898454053267 + 0.06414051364986957im
  -0.08699683911218295 + 0.18543610655059595im
    0.2161981237558695 - 0.0018863793562230863im
   0.08818777083472878 + 0.186775321019867im
  -0.11194407259046876 - 0.06209311450486262im
    0.0501488438189343 - 0.015403596908395312im
  0.007214393986800618 + 0.023268182399998488im
  -0.10421664974480917 - 0.0320709744271992im
   -0.1368615430095226 - 0.0010125543843854975im
                       ⋮
 -0.001970961353863288 + 0.01645767602593481im
  0.014208015819956503 + 0.002780449691514403im
  0.006550542874045706 - 0.021660350542753445im
  -0.03935671310771328 - 0.009364476890456762im
 0.0012967347651245673 + 0.002068807557763612im
 -0.010436641041897132 + 0.01932497009181391im
 -0.020949042838419255 - 0.00943219016616502im
  0.013127000647508597 + 0.030683449505479857im
  -0.012294

In [368]:
alm2 = Healpix.Alm(cs.lmax, cs.lmax, zeros(ComplexF64, Healpix.numberOfAlms(cs.lmax, cs.lmax)))
alm2.alm = alm_smooth[1,:]
mT = Healpix.alm2map(alm2, nside_in)   # 戻りは Healpix.Map

nalm  = Healpix.numberOfAlms(cs.lmax, cs.lmax)
almT = Healpix.Alm(cs.lmax, cs.lmax, zeros(ComplexF64, nalm))
almE = Healpix.Alm(cs.lmax, cs.lmax, zeros(ComplexF64, nalm))
almB = Healpix.Alm(cs.lmax, cs.lmax, zeros(ComplexF64, nalm))
almT.alm = alm_smooth[1,:]
almE.alm = alm_smooth[2,:]
almB.alm = alm_smooth[3,:]

maps= alm2map([almT, almE, almB], nside_in)

PolarizedHealpixMap{Float64, RingOrder, Vector{Float64}}([-58.49604137595402, -5.403537747114228, -11.298414133610322, -56.314667251724096, -59.699267217576754, -85.66622597795597, -71.17192250601221, -24.215049528928265, -74.16297882035632, -33.33706956041863  …  25.614428728708468, 142.1401271522759, 106.63577208567405, 222.1715373562783, 130.88070501569757, 156.216220113293, 62.39665256415908, 123.58634220400221, 167.54744250106972, 75.16966680268094], [0.3841684359262724, -0.2283283590288716, -0.588348244867616, -0.8973271051817646, -0.0369796019997603, 0.053658932027735334, -0.04304553931855282, -0.028340964839986604, -0.5793540400447466, 0.11914016822512552  …  0.034733190068783434, -0.4229562186824516, -0.01814141514429357, 0.036842700099326456, -0.20377726778227395, -0.06944079633454092, 0.4894117687322749, -0.43221459323236966, 0.013511342306182589, -0.2992332489751508], [-0.12281782625132895, 0.2541801993465586, 0.3691346888610124, -0.29191966435191946, 0.22773662495005553, 0

In [369]:
@show maps.i[pix_idx]
@show maps.q[pix_idx]
@show maps.u[pix_idx]

maps.i[pix_idx] = -85.66622597795597
maps.q[pix_idx] = 0.053658932027735334
maps.u[pix_idx] = 0.3644901510742654


0.3644901510742654

In [341]:
mT[pix_idx]

-85.66622597795597

In [308]:
mean(mT[1:4])

-32.878165127100665

In [309]:
x = Vector(@view alm_[1, :])
@show length(x)
@show size(A2_conj)

length(x) = 9216
size(A2_conj) = (9216, 474)


(9216, 474)

In [310]:
alm_[1,:]*A_conj

LoadError: DimensionMismatch: 

In [ ]:
blm_[:,1]

954-element Vector{ComplexF64}:
 0.28209479177387814 - 0.0im
                -0.0 - 0.0im
  0.4885756718694027 - 0.0im
                 0.0 - 0.0im
                 0.0 + 0.0im
                -0.0 - 0.0im
  0.6306791852123736 - 0.0im
                 0.0 - 0.0im
                 0.0 - 0.0im
                 0.0 + 0.0im
                -0.0 - 0.0im
  0.7461067059896269 - 0.0im
                 0.0 - 0.0im
                     ⋮
                 0.0 - 0.0im
                 0.0 - 0.0im
                 0.0 + 0.0im
                -0.0 - 0.0im
  2.0321911142347298 - 0.0im
                 0.0 - 0.0im
                 0.0 - 0.0im
                 0.0 + 0.0im
                -0.0 - 0.0im
   2.016251384998487 - 0.0im
                 0.0 - 0.0im
                 0.0 - 0.0im

In [ ]:
alm_

3×36864 Matrix{ComplexF64}:
 0.00724234+0.194644im  -0.933147+0.303313im  …   0.989512+0.106878im
   0.147042+0.363569im    1.05513+0.293518im      0.282053-0.563im
   0.302917+0.760422im  -0.921039-1.24593im      -0.553145+0.20714im

In [ ]:
A2_conj

36864×954 SparseMatrixCSC{ComplexF64, Int64} with 37240 stored entries:
⎡⣇⠀⎤
⎢⢸⠀⎥
⎢⢸⠀⎥
⎢⢸⠀⎥
⎢⢸⠀⎥
⎢⢸⠀⎥
⎢⠸⡄⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⢹⎥
⎢⠀⢸⎥
⎢⠀⢸⎥
⎢⠀⢸⎥
⎢⠀⢸⎥
⎢⠀⢸⎥
⎢⠀⢸⎥
⎣⠀⢸⎦

In [ ]:
A2_conj

36864×954 SparseMatrixCSC{ComplexF64, Int64} with 37240 stored entries:
⎡⣇⠀⎤
⎢⢸⠀⎥
⎢⢸⠀⎥
⎢⢸⠀⎥
⎢⢸⠀⎥
⎢⢸⠀⎥
⎢⠸⡄⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⡇⎥
⎢⠀⢹⎥
⎢⠀⢸⎥
⎢⠀⢸⎥
⎢⠀⢸⎥
⎢⠀⢸⎥
⎢⠀⢸⎥
⎢⠀⢸⎥
⎣⠀⢸⎦

In [ ]:
d = A2_conj*blm_[:,:]

36864×3 Matrix{ComplexF64}:
    0.282095+0.0im         …           0.0+0.0im
  0.00155815-0.00174369im              0.0+0.0im
     0.48856+0.0im                     0.0+0.0im
 -0.00155815-0.00174369im              0.0+0.0im
 -2.25171e-6-1.76104e-5im     -3.08838e-11+3.24047e-11im
   0.0034836-0.00389842im  …   -1.79357e-8-3.58021e-9im
    0.630619+0.0im             -1.15076e-6-4.38927e-6im
  -0.0034836-0.00389842im       0.00059938-0.000572579im
 -2.25171e-6+1.76104e-5im        0.0968065-0.00271672im
 -1.06749e-7-6.46566e-8im      7.87458e-14+5.37092e-13im
 -5.95626e-6-4.65828e-5im  …  -1.82675e-10+1.91672e-10im
  0.00582786-0.00652184im      -6.70957e-8-1.33933e-8im
    0.745965+0.0im             -3.04391e-6-1.16105e-5im
            ⋮              ⋱  
         0.0+0.0im                     0.0+0.0im
         0.0+0.0im                     0.0+0.0im
         0.0+0.0im                     0.0+0.0im
         0.0+0.0im         …           0.0+0.0im
         0.0+0.0im                     0.

In [72]:
A2_conj*blm_[:,3] == d[:,3]

true

In [69]:
d[2]

0.0015581475233141793 - 0.001743689657287808im

In [102]:
maximum(abs.(A_conj .- (A2_conj) ))

1.0153062434049297e-5

In [98]:
nside = 64
npix = 12*nside*nside

49152

In [99]:
sqrt(4pi/npix)

0.015989479811663883

In [96]:
cb.mmax

2

In [38]:
A_conj[17:19,17:19]

3×3 SparseMatrixCSC{ComplexF64, Int64} with 9 stored entries:
 -5.27432e-10-9.87154e-10im  …  1.52738e-14-4.69921e-15im
  -2.30294e-8-4.3029e-7im       6.56068e-12+2.11694e-12im
   3.95814e-5-9.13789e-5im       1.01428e-9+1.46518e-9im

In [ ]:
alm2map()

In [107]:
alm = Healpix.Alm(cs.lmax, cs.lmax, zeros(ComplexF64, Healpix.numberOfAlms(cs.lmax, cs.lmax)))


Alm{ComplexF64, Vector{ComplexF64}}(ComplexF64[0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im], 191, 191, 383)

In [109]:
alm.alm = almT
mT = Healpix.alm2map(alm, nside_in)   # 戻りは Healpix.Map

49152-element HealpixMap{Float64, RingOrder, Vector{Float64}}:
   38.96555299660666
  807.5809316309884
  -91.79553441193472
 -169.3918814040627
  -57.18339988536768
  -91.52479931272393
 -101.2656621807632
  892.4063680620143
 -267.04686585006357
 -301.29394392500404
  -58.19182146298763
  -32.96740109952867
   41.610278656112165
    ⋮
    5.21158594351253
    4.11483771838895
   -0.608098221973508
  -27.917653249985115
  -22.02779913541731
    8.247334519416194
    8.785795677008833
  -14.71976869807336
    4.2541379834573245
    8.503396105788195
   21.26340876673183
   -4.042358952116827

In [122]:
mean(mT[1:4])

146.33976720289942

In [144]:
mT

49152-element HealpixMap{Float64, RingOrder, Vector{Float64}}:
   38.96555299660666
  807.5809316309884
  -91.79553441193472
 -169.3918814040627
  -57.18339988536768
  -91.52479931272393
 -101.2656621807632
  892.4063680620143
 -267.04686585006357
 -301.29394392500404
  -58.19182146298763
  -32.96740109952867
   41.610278656112165
    ⋮
    5.21158594351253
    4.11483771838895
   -0.608098221973508
  -27.917653249985115
  -22.02779913541731
    8.247334519416194
    8.785795677008833
  -14.71976869807336
    4.2541379834573245
    8.503396105788195
   21.26340876673183
   -4.042358952116827

In [108]:
alm.alm

18528-element Vector{ComplexF64}:
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
     ⋮
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im